In [ ]:
import os
import json
import psycopg2
from app.db.connection import get_connection

from ingestion.loaders.ingestion import run_cards, run_decks
from ingestion.loaders.meta_blocks import META_BLOCKS


In [ ]:
# WARNING: This drops and recreates your DB.
# Use only during development.

!python scripts/reset_db.py


In [ ]:
!python scripts/migrate.py


In [ ]:
run_cards()


In [ ]:
run_decks()


In [ ]:
conn = get_connection()
cur = conn.cursor()

cur.execute("SELECT COUNT(*) AS count FROM decks;")
row = cur.fetchone()
print("Decks:", row["count"])

cur.execute("SELECT COUNT(*) AS count FROM deck_cards;")
row = cur.fetchone()
print("Deck Cards:", row["count"])

conn.close()



In [ ]:
conn = get_connection()
cur = conn.cursor()

cur.execute("""
    SELECT 
        id, 
        block_id, 
        deck_name, 
        author, 
        date, 
        country, 
        placement
    FROM decks
    ORDER BY id DESC
    LIMIT 10;
""")

rows = cur.fetchall()
rows


In [ ]:
deck_id = rows[0]["id"]  # dict access

cur = conn.cursor()
cur.execute("""
    SELECT card_id, quantity
    FROM deck_cards
    WHERE deck_id = %s
    ORDER BY card_id;
""", (deck_id,))

cur.fetchall()


In [ ]:
cur = conn.cursor()

cur.execute("""
    SELECT 
        d.deck_name,
        d.author,
        d.block_id,
        dc.card_id,
        dc.quantity,
        c.name AS card_name,
        c.color,
        c.type,
        c.level
    FROM decks d
    JOIN deck_cards dc ON dc.deck_id = d.id
    LEFT JOIN cards c ON c.card_id = dc.card_id
    WHERE d.id = %s
    ORDER BY dc.card_id;
""", (deck_id,))

cur.fetchall()


In [ ]:
cur = conn.cursor()

cur.execute("""
    SELECT DISTINCT dc.card_id
    FROM deck_cards dc
    LEFT JOIN cards c ON c.card_id = dc.card_id
    WHERE c.card_id IS NULL;
""")

missing = cur.fetchall()
missing


In [ ]:
import glob

raw_files = glob.glob("data/raw/decks/*.json")
raw_files[:10]


In [ ]:
with open(raw_files[0], "r", encoding="utf-8") as f:
    deck_json = json.load(f)

deck_json


In [ ]:
!python scripts/validate_decks.py


In [ ]:
!python scripts/export_decks.py


In [ ]:
import json

with open("data/processed/decks_export_flat.json", "r", encoding="utf-8") as f:
    decks = json.load(f)

len(decks), decks[0]


In [ ]:
# Collect all unique card IDs across all decks
card_vocab = sorted({
    card_id
    for deck in decks
    for card_id in deck["cards"].keys()
})

len(card_vocab), card_vocab[:20]


In [ ]:
import numpy as np

def deck_to_vector(deck, vocab):
    vec = np.zeros(len(vocab), dtype=np.float32)
    for i, card_id in enumerate(vocab):
        vec[i] = deck["cards"].get(card_id, 0)
    return vec

X = np.vstack([deck_to_vector(deck, card_vocab) for deck in decks])
X.shape



In [ ]:
X

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X)
similarity_matrix.shape


In [ ]:
import numpy as np

deck_index = 0  # choose the first deck
sims = similarity_matrix[deck_index]

# Top 10 most similar decks (excluding itself)
top_indices = sims.argsort()[::-1][1:11]

[(i, sims[i], decks[i]["deck_name"]) for i in top_indices]


In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer()
X_tfidf = tfidf.fit_transform(X).toarray()
X_tfidf.shape


In [ ]:
pip install umap-learn

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine")
X_umap = reducer.fit_transform(X_tfidf)

X_umap[:5]


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.scatter(X_umap[:, 0], X_umap[:, 1], s=10, alpha=0.7)
plt.title("Deck Clusters (UMAP Projection)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()


In [ ]:
plt.figure(figsize=(12, 10))
for i, deck in enumerate(decks):
    x, y = X_umap[i]
    plt.text(x, y, deck["deck_name"], fontsize=6, alpha=0.7)

plt.title("Deck Clusters with Labels")
plt.show()


In [ ]:
from sklearn.cluster import KMeans

k = 12  # adjust based on meta size
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(X_tfidf)

labels[:20]


In [ ]:
for deck, label in zip(decks, labels):
    deck["cluster"] = int(label)

decks[0]


In [ ]:
from collections import defaultdict

clusters = defaultdict(list)
for deck in decks:
    clusters[deck["cluster"]].append(deck["deck_name"])

clusters


In [ ]:
clusters[0]

In [ ]:
with open("data/processed/decks_with_clusters.json", "w", encoding="utf-8") as f:
    json.dump(decks, f, indent=2)



In [ ]:
from collections import defaultdict

cluster_to_decks = defaultdict(list)
for deck in decks:
    cluster_to_decks[deck["cluster"]].append(deck)

{c: len(v) for c, v in cluster_to_decks.items()}


In [ ]:
import numpy as np

cluster_card_freq = {}

for cluster, deck_list in cluster_to_decks.items():
    mat = np.vstack([deck_to_vector(deck, card_vocab) for deck in deck_list])
    freq = mat.sum(axis=0)  # total copies across all decks
    cluster_card_freq[cluster] = freq


In [ ]:
cluster_top_cards = {}

for cluster, freq in cluster_card_freq.items():
    sorted_idx = np.argsort(freq)[::-1]
    top_cards = [(card_vocab[i], int(freq[i])) for i in sorted_idx[:20]]
    cluster_top_cards[cluster] = top_cards

cluster_top_cards


In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

cluster_tfidf = {}

for cluster, deck_list in cluster_to_decks.items():
    mat = np.vstack([deck_to_vector(deck, card_vocab) for deck in deck_list])
    tfidf = TfidfTransformer().fit_transform(mat).toarray()
    importance = tfidf.sum(axis=0)
    cluster_tfidf[cluster] = importance


In [ ]:
cluster_signature_cards = {}

for cluster, importance in cluster_tfidf.items():
    sorted_idx = np.argsort(importance)[::-1]
    top_cards = [(card_vocab[i], float(importance[i])) for i in sorted_idx[:20]]
    cluster_signature_cards[cluster] = top_cards

cluster_signature_cards


In [ ]:
cluster_reports = {}

for cluster in cluster_to_decks:
    freq = cluster_top_cards[cluster]
    sig = cluster_signature_cards[cluster]

    cluster_reports[cluster] = {
        "top_cards": freq,
        "signature_cards": sig,
        "num_decks": len(cluster_to_decks[cluster])
    }

cluster_reports


In [ ]:
with open("data/processed/cluster_card_importance.json", "w", encoding="utf-8") as f:
    json.dump(cluster_reports, f, indent=2)


In [ ]:
import pandas as pd

for deck in decks:
    try:
        deck["date"] = pd.to_datetime(deck["date"])
    except:
        deck["date"] = pd.NaT

# Filter out decks with no date
decks_with_dates = [d for d in decks if d["date"] is not pd.NaT]

len(decks_with_dates)


In [ ]:
df = pd.DataFrame([
    {
        "id": d["id"],
        "deck_name": d["deck_name"],
        "cluster": d["cluster"],
        "date": d["date"],
        "block_id": d["block_id"],
        "placement": d["placement"],
        "country": d["country"]
    }
    for d in decks_with_dates
])

df.head()


In [ ]:
df["month"] = df["date"].dt.to_period("M")

popularity = df.groupby(["month", "cluster"]).size().reset_index(name="count")
popularity.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

for cluster in sorted(df["cluster"].unique()):
    sub = popularity[popularity["cluster"] == cluster]
    plt.plot(sub["month"].astype(str), sub["count"], label=f"Cluster {cluster}")

plt.xticks(rotation=45)
plt.title("Archetype Popularity Over Time")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def card_usage_over_time(card_id):
    rows = []
    for deck in decks_with_dates:
        qty = deck["cards"].get(card_id, 0)
        if qty > 0:
            rows.append({"date": deck["date"], "qty": qty})
    return pd.DataFrame(rows)

card_id = card_vocab[400]  # pick any card
usage = card_usage_over_time(card_id)
usage.head()


In [ ]:
if not usage.empty:
    usage["month"] = usage["date"].dt.to_period("M")
    trend = usage.groupby("month")["qty"].sum()

    plt.figure(figsize=(10, 6))
    plt.plot(trend.index.astype(str), trend.values)
    plt.title(f"Usage Trend for {card_id}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Card not used in any deck.")


In [ ]:
block_popularity = df.groupby(["block_id", "cluster"]).size().reset_index(name="count")
block_popularity


In [ ]:
plt.figure(figsize=(12, 8))

for cluster in sorted(df["cluster"].unique()):
    sub = block_popularity[block_popularity["cluster"] == cluster]
    plt.plot(sub["block_id"], sub["count"], marker="o", label=f"Cluster {cluster}")

plt.xticks(rotation=45)
plt.title("Archetype Popularity by Block")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import re
import numpy as np

def parse_placement(value):
    if value is None:
        return np.nan
    if isinstance(value, (int, float)):
        return value

    s = str(value).strip().lower()

    # Champion = 1
    if "champion" in s or "winner" in s:
        return 1

    # T4, T8, T16, etc.
    m = re.match(r"t\s*(\d+)", s)
    if m:
        return int(m.group(1))

    # Top X → X
    m = re.match(r"top\s*(\d+)", s)
    if m:
        return int(m.group(1))

    # 1st, 2nd, 3rd, 4th → number
    m = re.match(r"(\d+)", s)
    if m:
        return int(m.group(1))

    return np.nan

df["placement_num"] = df["placement"].apply(parse_placement)
df[["placement", "placement_num"]].head(20)



In [ ]:
placement_df = df[df["placement"].notnull()]

placement_summary = placement_df.groupby("cluster")["placement_num"].mean().sort_values()
placement_summary


In [ ]:
archetype_schema = {
    "sukamon_etemon": {
        "name": "Sukamon/Etemon",
        "keywords": ["sukamon", "chuumon", "etemon", "suka"],
        "category": "Rookie Rush / Meme"
    },
    "alphamon": {
        "name": "Alphamon",
        "keywords": ["alphamon", "ouryumon", "grademon", "ouryuken", "dourgreymon"],
        "category": "Black / X-Antibody"
    },
    "cs_alpha": {
        "name": "CS Alphamon",
        "keywords": ["kyoko", "takumi", "palmon", "terriermon", "hagurumon", "alphamon"],
        "category": "Cyber Sleuth"
    },
    # … you will paste the full cleaned list here …
}


In [ ]:
import psycopg2
import psycopg2.extras

def load_card_db():
    conn = psycopg2.connect(
        dbname="digimon",
        user="postgres",
        password="CHANGE_ME",
        host="localhost",
        port=5432
    )
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)

    cur.execute("SELECT card_id, name FROM cards;")
    rows = cur.fetchall()

    conn.close()

    return {row["card_id"]: row["name"] for row in rows}

card_db = load_card_db()
len(card_db)


In [ ]:
def load_tamer_db():
    conn = psycopg2.connect(
        dbname="digimon",
        user="postgres",
        password="CHANGE_ME",
        host="localhost",
        port=5432
    )
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)

    cur.execute("""
        SELECT card_id, name 
        FROM cards
        WHERE type ILIKE '%tamer%';
    """)

    rows = cur.fetchall()
    conn.close()

    return {row["card_id"]: row["name"] for row in rows}

tamer_db = load_tamer_db()


In [ ]:
card_db

In [ ]:
def deck_card_names(deck, card_db):
    names = []
    for card_id in deck["cards"].keys():
        name = card_db.get(card_id)
        if name:
            names.append(name.lower())
    return names


In [ ]:
def deck_tamer_names(deck, tamer_db):
    names = []
    for card_id in deck["cards"].keys():
        if card_id in tamer_db:
            names.append(tamer_db[card_id].lower())
    return names


In [ ]:
from rapidfuzz import fuzz

def fuzzy_match(a, b, threshold=80):
    """Return True if strings match above threshold."""
    return fuzz.partial_ratio(a.lower(), b.lower()) >= threshold


In [ ]:
def match_archetype_from_schema(deck, schema, card_db, tamer_db):
    card_names = deck_card_names(deck, card_db)
    tamer_names = deck_tamer_names(deck, tamer_db)

    best_label = None
    best_score = 0

    for key, info in schema.items():
        score = 0

        # 1. Fuzzy match Digimon names
        for kw in info["keywords"]:
            kw = kw.lower()
            for name in card_names:
                if fuzzy_match(kw, name):
                    score += 1

        # 2. Fuzzy match tamers (weighted)
        for kw in info["keywords"]:
            kw = kw.lower()
            for tname in tamer_names:
                if fuzzy_match(kw, tname):
                    score += 3   # tamers count triple

        if score > best_score:
            best_score = score
            best_label = info["name"]

    return best_label, best_score



In [ ]:
for deck in decks:
    label, score = match_archetype_from_schema(deck, archetype_schema)
    deck["schema_archetype"] = label
    deck["schema_score"] = score

decks[0]


In [ ]:
for deck in decks:
    if deck["schema_score"] >= 2:  # threshold you can tune
        deck["final_archetype"] = deck["schema_archetype"]
    else:
        deck["final_archetype"] = deck["archetype"]

decks[0]


In [ ]:
with open("data/archetypes.json", "r", encoding="utf-8") as f:
    archetype_schema = json.load(f)


In [ ]:
for deck in decks:
    label, score = match_archetype_from_schema(deck, archetype_schema)
    deck["schema_archetype"] = label
    deck["schema_score"] = score


In [ ]:
with open("data/processed/decks_final_labeled.json", "w", encoding="utf-8") as f:
    json.dump(decks, f, indent=2)


In [ ]:
import json
import re
from pathlib import Path

raw_path = Path("data/archetypes_raw.txt")
out_path = Path("data/archetypes.json")

def slugify(name: str) -> str:
    s = name.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

archetypes = {}

with raw_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        if ":" not in line:
            continue

        # Split "Name: a, b, c"
        name_part, rest = line.split(":", 1)
        name = name_part.strip()
        keywords = [k.strip() for k in rest.split(",") if k.strip()]

        archetype_id = slugify(name)

        archetypes[archetype_id] = {
            "name": name,
            "keywords": keywords,
            "category": "",
            "variants": [],
            "notes": ""
        }

with out_path.open("w", encoding="utf-8") as f:
    json.dump(archetypes, f, indent=2, ensure_ascii=False)

out_path, len(archetypes)


In [ ]:
def match_variant(deck, archetype_info, card_db, tamer_db):
    card_names = deck_card_names(deck, card_db)
    tamer_names = deck_tamer_names(deck, tamer_db)

    best_variant = None
    best_score = 0

    for variant in archetype_info.get("variants", []):
        score = 0

        for kw in variant.get("keywords", []):
            kw = kw.lower()

            # Digimon fuzzy match
            for name in card_names:
                if fuzzy_match(kw, name):
                    score += 1

            # Tamer fuzzy match (weighted)
            for tname in tamer_names:
                if fuzzy_match(kw, tname):
                    score += 3

        if score > best_score:
            best_score = score
            best_variant = variant["name"]

    return best_variant, best_score




In [ ]:
archetype_schema

In [ ]:
for deck in decks:
    archetype, score = match_archetype_from_schema(deck, archetype_schema, card_db, tamer_db)
    deck["archetype"] = archetype

    if archetype:
        archetype_id = slugify(archetype)
        variant, vscore = match_variant(deck, archetype_schema[archetype_id], card_db, tamer_db)
        deck["variant"] = variant if vscore > 0 else None



In [ ]:
decks[1000]

In [ ]:
decks[1]["archetype"]

In [ ]:
for deck in decks:
    if deck["variant"] is not None:
        print(deck)

In [ ]:
pip install rapidfuzz